In [12]:
import numpy as np
import pandas as pd
import pickle
import logging
from pathlib import Path
from sklearn.metrics import adjusted_rand_score
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s — %(levelname)s — %(message)s",
)
logger = logging.getLogger(__name__)

RESULTS_DIR = Path("results")

In [13]:
# Load & validate all result files
#
# We check every key we depend on before doing any analysis.
# A missing or malformed pickle is caught here with a clear message
# rather than a cryptic KeyError three cells later.

REQUIRED_KEYS = {
    "not_weighted_results.pkl": [
        "summary_df", "winner", "winner_k", "winner_sil",
        "winner_lbls", "all_labels", "all_metrics",
    ],
    "weighted_results.pkl": [
        "summary_df", "winner", "best_alpha", "winner_k", "winner_sil",
        "winner_lbls", "df_sweep", "df_best", "best_gower",
    ],
    "umap_results.pkl": [
        "umap_lbls", "umap_k", "umap_sil", "umap_stability",
        "config_used", "dist_used", "alpha_used", "emb2d",
        "umap_results", "stability_scores",
    ],
}

loaded = {}
all_ok = True

for filename, required_keys in REQUIRED_KEYS.items():
    path = RESULTS_DIR / filename
    logger.info("── Checking %s ──", filename)

    # File exists
    if not path.exists():
        logger.error("MISSING FILE: %s — run the corresponding notebook first.", path)
        all_ok = False
        continue

    # Loads without error
    try:
        with open(path, "rb") as f:
            data = pickle.load(f)
        logger.info("  OK  loaded successfully")
    except Exception as e:
        logger.error("  FAIL  could not load: %s", e)
        all_ok = False
        continue

    # All required keys present
    missing_keys = [k for k in required_keys if k not in data]
    if missing_keys:
        logger.error("  FAIL  missing keys: %s", missing_keys)
        all_ok = False
        continue
    logger.info("  OK  all required keys present")

    # Store
    loaded[filename] = data

if not all_ok:
    raise RuntimeError(
        "One or more result files are missing or malformed. "
        "Fix the errors above before continuing."
    )

nw = loaded["not_weighted_results.pkl"]
w  = loaded["weighted_results.pkl"]
u  = loaded["umap_results.pkl"]

logger.info("All result files loaded successfully.")

2026-03-20 11:42:31,279 — INFO — ── Checking not_weighted_results.pkl ──
2026-03-20 11:42:31,388 — INFO —   OK  loaded successfully
2026-03-20 11:42:31,389 — INFO —   OK  all required keys present
2026-03-20 11:42:31,389 — INFO — ── Checking weighted_results.pkl ──
2026-03-20 11:42:31,397 — INFO —   OK  loaded successfully
2026-03-20 11:42:31,397 — INFO —   OK  all required keys present
2026-03-20 11:42:31,397 — INFO — ── Checking umap_results.pkl ──
2026-03-20 11:42:31,409 — INFO —   OK  loaded successfully
2026-03-20 11:42:31,409 — INFO —   OK  all required keys present
2026-03-20 11:42:31,421 — INFO — All result files loaded successfully.


In [14]:
# Validate label arrays
#
# Before computing ARI we check that all label arrays:
#   - are numpy integer arrays
#   - have the same length (same number of clients)
#   - have at least 2 unique clusters
#   - do not contain NaN or negative values
#
# We also log the cluster count and size distribution for each,
# so any obvious problem (e.g. one method producing k=1) is visible.

def validate_labels(lbls: np.ndarray, name: str) -> bool:
    ok = True
    lbls = np.asarray(lbls)

    if not np.issubdtype(lbls.dtype, np.integer):
        logger.error("[%s] FAIL dtype: expected integer, got %s", name, lbls.dtype)
        ok = False
    else:
        logger.info("[%s] OK   dtype: %s", name, lbls.dtype)

    if np.any(lbls < 0):
        logger.error("[%s] FAIL negative labels found", name)
        ok = False
    else:
        logger.info("[%s] OK   no negative labels", name)

    n_unique = len(np.unique(lbls))
    if n_unique < 2:
        logger.error("[%s] FAIL only %d unique cluster(s) — degenerate result", name, n_unique)
        ok = False
    else:
        logger.info("[%s] OK   %d unique clusters", name, n_unique)

    sizes = pd.Series(lbls).value_counts().sort_index()
    min_size = sizes.min()
    max_size = sizes.max()
    if min_size < 3:
        logger.warning(
            "[%s] WARN smallest cluster has %d members — "
            "may be an outlier cluster rather than a genuine segment",
            name, min_size,
        )
    logger.info("[%s] cluster sizes: min=%d  max=%d  distribution=%s",
                name, min_size, max_size, sizes.to_dict())

    return ok

label_sets = {
    "Not weighted": np.asarray(nw["winner_lbls"]),
    "Weighted":     np.asarray(w["winner_lbls"]),
    "UMAP":         np.asarray(u["umap_lbls"]),
}

logger.info("── Label array validation ──")
label_ok = True
n_ref = len(label_sets["Not weighted"])

for name, lbls in label_sets.items():
    if len(lbls) != n_ref:
        logger.error(
            "[%s] FAIL length mismatch: got %d, expected %d — "
            "notebooks may have been run on different data subsets",
            name, len(lbls), n_ref,
        )
        label_ok = False
    label_ok &= validate_labels(lbls, name)

if not label_ok:
    raise RuntimeError(
        "Label validation failed. Ensure all three notebooks were run "
        "on the same preprocessed dataset."
    )

logger.info("All label arrays valid — %d clients, 3 label sets.", n_ref)

2026-03-20 11:42:31,447 — INFO — ── Label array validation ──
2026-03-20 11:42:32,106 — INFO — [Not weighted] OK   dtype: uint64
2026-03-20 11:42:32,106 — INFO — [Not weighted] OK   no negative labels
2026-03-20 11:42:32,109 — INFO — [Not weighted] OK   10 unique clusters
2026-03-20 11:42:32,109 — INFO — [Not weighted] cluster sizes: min=228  max=1314  distribution={0: 255, 1: 286, 2: 313, 3: 468, 4: 228, 5: 1314, 6: 1248, 7: 249, 8: 360, 9: 229}
2026-03-20 11:42:32,109 — INFO — [Weighted] OK   dtype: uint64
2026-03-20 11:42:32,109 — INFO — [Weighted] OK   no negative labels
2026-03-20 11:42:32,109 — INFO — [Weighted] OK   10 unique clusters
2026-03-20 11:42:32,119 — INFO — [Weighted] cluster sizes: min=228  max=1314  distribution={0: 256, 1: 286, 2: 313, 3: 468, 4: 249, 5: 1314, 6: 1248, 7: 359, 8: 228, 9: 229}
2026-03-20 11:42:32,120 — INFO — [UMAP] OK   dtype: uint64
2026-03-20 11:42:32,121 — INFO — [UMAP] OK   no negative labels
2026-03-20 11:42:32,121 — INFO — [UMAP] OK   10 uniqu

In [15]:
# Validate ARI preconditions
#
# ARI is only meaningful when comparing clusterings of the same objects.
# We verify that:
#   1. All label arrays have identical length
#   2. No array is trivial (k=1 or k=n)
#   3. Log a warning if any two methods have very different k values —
#      large k differences structurally lower ARI even when the methods
#      agree on the broad groupings

logger.info("── ARI precondition checks ──")

n_clients = n_ref
for name, lbls in label_sets.items():
    k = len(np.unique(lbls))
    if k == n_clients:
        logger.error(
            "[%s] FAIL k=n_clients (%d) — every client is its own cluster, "
            "ARI will be 0 by construction", name, k,
        )
    elif k == 1:
        logger.error(
            "[%s] FAIL k=1 — all clients in one cluster, "
            "ARI will be 0 by construction", name,
        )
    else:
        logger.info("[%s] OK  k=%d", name, k)

k_values = {name: len(np.unique(lbls)) for name, lbls in label_sets.items()}
k_list   = list(k_values.values())
k_range  = max(k_list) - min(k_list)

if k_range > 4:
    logger.warning(
        "k values differ substantially across methods: %s — "
        "ARI will be structurally penalised because one method splits "
        "groups that another merges. A low ARI may reflect k disagreement "
        "rather than genuinely different client assignments.",
        k_values,
    )
else:
    logger.info("k values are reasonably aligned: %s", k_values)

2026-03-20 11:42:32,136 — INFO — ── ARI precondition checks ──
2026-03-20 11:42:32,137 — INFO — [Not weighted] OK  k=10
2026-03-20 11:42:32,138 — INFO — [Weighted] OK  k=10
2026-03-20 11:42:32,139 — INFO — [UMAP] OK  k=10
2026-03-20 11:42:32,141 — INFO — k values are reasonably aligned: {'Not weighted': 10, 'Weighted': 10, 'UMAP': 10}


In [16]:
# Cross-notebook silhouette summary
#
# NOTE: the three silhouette values are computed in different spaces and
# cannot be compared in magnitude. They are shown together only for
# completeness. The meaningful comparison is the ARI in Cell 6.

rows = [
    {
        "Notebook":        "Not weighted",
        "Config":          nw["winner"],
        "Alpha":           "neutral",
        "k":               nw["winner_k"],
        "Silhouette":      round(nw["winner_sil"], 4),
        "Space":           "precomputed distance",
        "Comparable?":     "baseline",
    },
    {
        "Notebook":        "Weighted",
        "Config":          w["winner"],
        "Alpha":           round(w["best_alpha"], 4) if w["best_alpha"] else "—",
        "k":               w["winner_k"],
        "Silhouette":      round(w["winner_sil"], 4),
        "Space":           "precomputed distance",
        "Comparable?":     "yes — same space as not weighted",
    },
    {
        "Notebook":        "UMAP",
        "Config":          f"{u['dist_used']} / {u['config_used']}",
        "Alpha":           round(u["alpha_used"], 4) if u["alpha_used"] else "—",
        "k":               u["umap_k"],
        "Silhouette":      round(u["umap_sil"], 4),
        "Space":           "UMAP latent (Euclidean)",
        "Comparable?":     "NO — different space",
    },
]

summary_df = pd.DataFrame(rows)
print("\n── Cross-notebook summary ──")
print("NOTE: silhouette scores in different spaces cannot be compared in magnitude.")
display(summary_df)

# Improvement from not-weighted to weighted (same space — valid comparison)
delta_nw_to_w = w["winner_sil"] - nw["winner_sil"]
logger.info(
    "Weighted vs not-weighted improvement: %+.4f  (%s)",
    delta_nw_to_w,
    "weighted better" if delta_nw_to_w > 0 else "not-weighted better or equal",
)
print(f"\nWeighted vs not-weighted (same space, valid): {delta_nw_to_w:+.4f}")


── Cross-notebook summary ──
NOTE: silhouette scores in different spaces cannot be compared in magnitude.


,Notebook,Config,Alpha,k,Silhouette,Space,Comparable?
0,Not weighted,Mixed (L1+Tanimoto),neutral,10,0.4577,precomputed distance,baseline
1,Weighted,L1+Tanimoto,0.2,10,0.6788,precomputed distance,yes — same space as not weighted
2,UMAP,L1+Tanimoto / umap_8d,0.2,10,0.7813,UMAP latent (Euclidean),NO — different space


2026-03-20 11:42:32,161 — INFO — Weighted vs not-weighted improvement: +0.2211  (weighted better)



Weighted vs not-weighted (same space, valid): +0.2211


In [17]:
# Adjusted Rand Index matrix
#
# ARI measures agreement between two clusterings regardless of the space
# they were computed in. Range: [-1, 1]. Expected value under random
# assignment is 0. Interpretation guide:
#
#   >= 0.85  near-perfect agreement
#   0.60-0.85  strong agreement
#   0.40-0.60  partial agreement
#   0.00-0.40  weak agreement
#   < 0.00   worse than random

pairs     = list(label_sets.items())
n         = len(pairs)
ari_matrix = np.zeros((n, n))

for i, (_, li) in enumerate(pairs):
    for j, (_, lj) in enumerate(pairs):
        ari_matrix[i, j] = adjusted_rand_score(li, lj)

names  = [p[0] for p in pairs]
ari_df = pd.DataFrame(ari_matrix, index=names, columns=names).round(4)

print("\n── Adjusted Rand Index matrix ──")
display(ari_df)

for i in range(n):
    for j in range(i + 1, n):
        ari_val = ari_matrix[i, j]
        if ari_val >= 0.85:
            interp = "near-perfect agreement"
        elif ari_val >= 0.60:
            interp = "strong agreement"
        elif ari_val >= 0.40:
            interp = "partial agreement"
        elif ari_val >= 0.00:
            interp = "weak agreement"
        else:
            interp = "worse than random — investigate"
        logger.info("ARI %s vs %s: %.4f → %s",
                    names[i], names[j], ari_val, interp)
        print(f"  {names[i]:15s} vs {names[j]:15s}: ARI={ari_val:.4f}  ({interp})")

fig_ari = px.imshow(
    ari_df,
    color_continuous_scale="Viridis",
    zmin=0, zmax=1,
    text_auto=".3f",
    title="Adjusted Rand Index — agreement between clustering approaches",
)
fig_ari.update_layout(height=400)
fig_ari.show()


── Adjusted Rand Index matrix ──


,Not weighted,Weighted,UMAP
Not weighted,1.0000,0.9998,0.8450
Weighted,0.9998,1.0000,0.8451
UMAP,0.8450,0.8451,1.0000


2026-03-20 11:42:32,195 — INFO — ARI Not weighted vs Weighted: 0.9998 → near-perfect agreement
2026-03-20 11:42:32,197 — INFO — ARI Not weighted vs UMAP: 0.8450 → strong agreement
2026-03-20 11:42:32,198 — INFO — ARI Weighted vs UMAP: 0.8451 → strong agreement


  Not weighted    vs Weighted       : ARI=0.9998  (near-perfect agreement)
  Not weighted    vs UMAP           : ARI=0.8450  (strong agreement)
  Weighted        vs UMAP           : ARI=0.8451  (strong agreement)


### ARI matrix

All three methods are in strong-to-near-perfect agreement:

- Not weighted vs weighted: ARI = 0.988 — near-identical assignments despite different alpha values. The alpha sweep moved from k=10 neutral to k=10 optimal but did not fundamentally reorganise which clients belong together.
- Not weighted vs UMAP: ARI = 0.825 — strong agreement between a distance-space and a latent-space method.
- Weighted vs UMAP: ARI = 0.824 — essentially the same level of agreement.

The UMAP result is an independent validation: a method that operates in a completely different geometric space, with different assumptions, finds the same client groups. This substantially increases confidence that the segments reflect real structure in the data rather than artefacts of the distance function.


In [18]:
# Alpha sweep summary: what did weighting add?
#
# This shows whether the alpha sweep improved over the neutral baseline,
# and where the peak silhouette sat relative to neutral alpha.

print("\n── Alpha sweep results (weighted notebook) ──")
display(w["df_best"])

# Neutral baselines (recalculated for reference)
# These match what the weighted notebook computed
n_num_cols = 14
n_cat_ham  = 3
n_cat_tan  = 10
neutral_hamming  = n_num_cols / (n_num_cols + n_cat_ham)
neutral_tanimoto = n_num_cols / (n_num_cols + n_cat_tan)

fig_sweep = go.Figure()

for _, row in w["df_best"].iterrows():
    config_label = f"{row['num_dist']}+{row['cat_dist']}"
    fig_sweep.add_trace(go.Bar(
        x=[config_label],
        y=[row["mean_sil"]],
        error_y=dict(type="data", array=[row["std_sil"]], visible=True),
        name=config_label,
        text=f"α={row['alpha']:.2f}<br>k={row['k']}",
        textposition="outside",
    ))

fig_sweep.add_hline(
    y=nw["winner_sil"],
    line_dash="dash", line_color="gray",
    annotation_text=f"Not weighted best ({nw['winner_sil']:.3f})",
    annotation_position="top left",
)
fig_sweep.update_layout(
    title="Weighted sweep — best mean silhouette per config vs not-weighted baseline",
    yaxis_title="Mean silhouette (5 seeds)",
    xaxis_title="Distance config",
    height=500,
    showlegend=False,
)
fig_sweep.show()

# Gower reference
gower_sil = w["best_gower"]["mean_sil"]
print(f"\nGower baseline     : sil={gower_sil:.4f}  k={w['best_gower']['k']}")
print(f"Not weighted best  : sil={nw['winner_sil']:.4f}  k={nw['winner_k']}  config={nw['winner']}")
print(f"Weighted best      : sil={w['winner_sil']:.4f}  k={w['winner_k']}  config={w['winner']}  α={w['best_alpha']:.4f}")


── Alpha sweep results (weighted notebook) ──


,num_dist,cat_dist,alpha,k,mean_sil,std_sil
0,L1,Tanimoto,0.2,10,0.6788,0.0
1,Canberra,Tanimoto,0.2,10,0.6671,0.0
2,L2,Tanimoto,0.2,10,0.6579,0.0
3,L1,Hamming,0.2,10,0.6492,0.0
4,Canberra,Hamming,0.2,10,0.6309,0.0
5,L2,Hamming,0.2,10,0.6216,0.0



Gower baseline     : sil=0.1656  k=4
Not weighted best  : sil=0.4577  k=10  config=Mixed (L1+Tanimoto)
Weighted best      : sil=0.6788  k=10  config=L1+Tanimoto  α=0.2000


In [19]:
# UMAP stability summary
print("\n── UMAP stability across seeds ──")
stab_rows = []
for config_name, ari in u["stability_scores"].items():
    res = u["umap_results"][config_name]
    stab_rows.append({
        "Config":      config_name,
        "k":           res["best_k"],
        "Silhouette":  res["best_score"],
        "Stability ARI": round(ari, 4),
        "Stable?":     "YES" if ari >= 0.85 else "NO",
    })

stab_df = pd.DataFrame(stab_rows).sort_values("Stability ARI", ascending=False)
display(stab_df)

fig_stab = go.Figure()
colors = [
    px.colors.qualitative.Vivid[0] if s >= 0.85
    else px.colors.qualitative.Vivid[2]
    for s in u["stability_scores"].values()
]
fig_stab.add_trace(go.Bar(
    x=list(u["stability_scores"].keys()),
    y=list(u["stability_scores"].values()),
    marker_color=colors,
    text=[f"{v:.3f}" for v in u["stability_scores"].values()],
    textposition="outside",
))
fig_stab.add_hline(
    y=0.85,
    line_dash="dash", line_color="red",
    annotation_text="stability threshold (0.85)",
    annotation_position="top right",
)
fig_stab.update_layout(
    title="UMAP stability ARI per embedding configuration",
    yaxis_title="Mean ARI across seed pairs",
    yaxis_range=[0, 1.05],
    height=420,
    showlegend=False,
)
fig_stab.show()


── UMAP stability across seeds ──


,Config,k,Silhouette,Stability ARI,Stable?
2,umap_15d,10,0.759259,0.9538,YES
0,umap_8d,10,0.781307,0.9477,YES
1,umap_12d,10,0.757731,0.9330,YES


In [20]:
# Same UMAP embedding, three sets of labels
#
# Projects all three methods' cluster labels onto the UMAP 2D embedding.
# Any visual difference between subplots is due purely to label assignment —
# the layout is identical. This makes it easy to see where methods agree
# and where they diverge.

emb2d = u["emb2d"]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        f"Not weighted (k={nw['winner_k']})",
        f"Weighted (k={w['winner_k']})",
        f"UMAP (k={u['umap_k']})",
    ],
    horizontal_spacing=0.05,
)

for col_idx, (approach, lbls) in enumerate([
    ("Not weighted", label_sets["Not weighted"]),
    ("Weighted",     label_sets["Weighted"]),
    ("UMAP",         label_sets["UMAP"]),
]):
    for cluster_id in sorted(np.unique(lbls)):
        mask = lbls == cluster_id
        fig.add_trace(
            go.Scatter(
                x=emb2d[mask, 0], y=emb2d[mask, 1],
                mode="markers",
                marker=dict(
                    size=5,
                    color=px.colors.qualitative.Vivid[
                        cluster_id % len(px.colors.qualitative.Vivid)
                    ],
                    opacity=0.7,
                ),
                name=f"Cluster {cluster_id}",
                legendgroup=f"{approach}_{cluster_id}",
                showlegend=(col_idx == 0),
            ),
            row=1, col=col_idx + 1,
        )

fig.update_layout(
    title="Same UMAP embedding — coloured by each method's cluster labels",
    height=520,
)
fig.show()

The three scatter plots use the same 2D embedding — only the colour assignment changes. The visual structure is consistent across all three panels: the same point clouds appear in the same positions, and the colour boundaries between them shift only slightly. Not weighted and weighted are nearly indistinguishable visually, which is consistent with ARI = 0.988. The UMAP panel (k=6) shows some of the finer clusters from the k=10 solutions merged into single colours — particularly the dense groups in the upper right and centre — reflecting the fact that UMAP found 6 broader segments where the distance-space methods found 10.


In [21]:
# Pairwise client agreement heatmap
#
# For each pair of methods, we compute a co-assignment matrix:
# what fraction of client pairs that are in the same cluster under
# method A are also in the same cluster under method B.
# This reveals whether the methods agree on the CORE of each cluster
# even when they disagree on the exact boundaries.

def coassignment_matrix(lbls_a: np.ndarray, lbls_b: np.ndarray) -> pd.DataFrame:
    """
    For each cluster in A, show what fraction of its members land in
    each cluster of B. Rows = A clusters, columns = B clusters.
    Dominated rows (one cell >> others) = high agreement on that group.
    Spread rows = A splits a group that B merges, or vice versa.
    """
    ka = sorted(np.unique(lbls_a))
    kb = sorted(np.unique(lbls_b))
    mat = np.zeros((len(ka), len(kb)))
    for i, a in enumerate(ka):
        mask = lbls_a == a
        total = mask.sum()
        for j, b in enumerate(kb):
            mat[i, j] = (lbls_b[mask] == b).sum() / total
    return pd.DataFrame(mat,
                        index=[f"A{a}" for a in ka],
                        columns=[f"B{b}" for b in kb])

pairs_to_plot = [
    ("Not weighted", "Weighted",
     label_sets["Not weighted"], label_sets["Weighted"]),
    ("Not weighted", "UMAP",
     label_sets["Not weighted"], label_sets["UMAP"]),
    ("Weighted",     "UMAP",
     label_sets["Weighted"],     label_sets["UMAP"]),
]

for name_a, name_b, lbls_a, lbls_b in pairs_to_plot:
    co = coassignment_matrix(lbls_a, lbls_b)
    fig = px.imshow(
        co,
        color_continuous_scale="Blues",
        zmin=0, zmax=1,
        text_auto=".2f",
        title=f"Co-assignment: {name_a} (rows) → {name_b} (cols)",
        labels={"x": name_b + " cluster", "y": name_a + " cluster",
                "color": "fraction"},
    )
    fig.update_layout(height=400)
    fig.show()

The co-assignment matrices confirm what the ARI summarises. In both heatmaps (not weighted → weighted and not weighted → UMAP) each row has a single dominant dark cell with value ≥ 0.91, and near-zero values everywhere else. This means every not-weighted cluster maps cleanly to exactly one cluster in both the weighted and UMAP solutions — there is no splitting or merging ambiguity.

The slight imperfections (A0 → UMAP B0 at 0.94 with 0.06 leaking to B1, A9 → UMAP B5 at 0.91 with 0.09 elsewhere) are boundary clients — a small number of observations that sit between clusters and are assigned differently depending on the method. These are the clients worth inspecting individually in a business context, as they may represent genuinely ambiguous profiles.


In [22]:
# Final verdict
print("=" * 60)
print("FINAL VERDICT")
print("=" * 60)

ari_nw_w   = ari_matrix[0, 1]
ari_nw_u   = ari_matrix[0, 2]
ari_w_u    = ari_matrix[1, 2]

delta_nw_to_w = w["winner_sil"] - nw["winner_sil"]
delta_gower   = nw["winner_sil"] - gower_sil

print(f"""
Distance-space results (comparable — same space)
  Gower baseline          : sil={gower_sil:.4f}  k={w['best_gower']['k']}
  Not weighted best       : sil={nw['winner_sil']:.4f}  k={nw['winner_k']}  config={nw['winner']}
  Weighted best           : sil={w['winner_sil']:.4f}  k={w['winner_k']}  config={w['winner']}  α={w['best_alpha']:.4f}

  Custom distance vs Gower  : {nw['winner_sil'] - gower_sil:+.4f}
  Weighted vs not weighted  : {delta_nw_to_w:+.4f}

UMAP results (not comparable in magnitude — different space)
  Best config             : {u['dist_used']} / {u['config_used']}
  k={u['umap_k']}  sil={u['umap_sil']:.4f}  stability ARI={u['umap_stability']:.3f}

Agreement between methods (ARI)
  Not weighted vs weighted : {ari_nw_w:.4f}
  Not weighted vs UMAP     : {ari_nw_u:.4f}
  Weighted vs UMAP         : {ari_w_u:.4f}
""")

# Automated conclusion
if ari_w_u >= 0.70:
    conclusion = (
        "Weighted distance-space and UMAP clusterings strongly agree. "
        "Report weighted as primary (directly interpretable in feature space), "
        "cite UMAP as independent validation."
    )
elif ari_w_u >= 0.40:
    conclusion = (
        "Partial agreement between weighted and UMAP. "
        "Core segments are consistent. Boundary clients differ — "
        "inspect the co-assignment matrix (Cell 10) to identify them."
    )
else:
    conclusion = (
        "Weak agreement between weighted and UMAP. "
        "Methods find different structure. Do not report either as final "
        "without further investigation."
    )

print(f"Conclusion:\n  {conclusion}")
logger.info("Final verdict: ARI weighted vs UMAP = %.4f — %s", ari_w_u, conclusion)

2026-03-20 11:42:32,634 — INFO — Final verdict: ARI weighted vs UMAP = 0.8451 — Weighted distance-space and UMAP clusterings strongly agree. Report weighted as primary (directly interpretable in feature space), cite UMAP as independent validation.


FINAL VERDICT

Distance-space results (comparable — same space)
  Gower baseline          : sil=0.1656  k=4
  Not weighted best       : sil=0.4577  k=10  config=Mixed (L1+Tanimoto)
  Weighted best           : sil=0.6788  k=10  config=L1+Tanimoto  α=0.2000

  Custom distance vs Gower  : +0.2921
  Weighted vs not weighted  : +0.2211

UMAP results (not comparable in magnitude — different space)
  Best config             : L1+Tanimoto / umap_8d
  k=10  sil=0.7813  stability ARI=0.948

Agreement between methods (ARI)
  Not weighted vs weighted : 0.9998
  Not weighted vs UMAP     : 0.8450
  Weighted vs UMAP         : 0.8451

Conclusion:
  Weighted distance-space and UMAP clusterings strongly agree. Report weighted as primary (directly interpretable in feature space), cite UMAP as independent validation.


### Conclusion

The three methods converge. The weighted distance-space result is the primary segmentation. UMAP confirms it independently. The boundary disagreement is minimal and localised to a small number of clients on cluster edges.